# Ecuador Under-5 Mortality Rate Analysis
## Synthetic Control Evaluation of World Bank Interventions (1992)

This notebook applies synthetic control methodology to evaluate whether 
World Bank development interventions in Ecuador accelerated reductions 
in under-5 mortality beyond regional trends.

In [72]:
!pip install wbgapi



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [73]:
!pip install pysyncon


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [74]:
import wbgapi as wb
import pandas as pd
from pysyncon import Dataprep, Synth
from pysyncon.utils import PlaceboTest
import matplotlib.pyplot as plt
import matplotlib

## Data
Sourced directly from World Bank Development Indicators API via wbgapi.
No local files required. Indicators: U5MR, fertility rate, GDP per capita 
(constant 2015 USD), DPT immunization coverage.

In [75]:
queries = [
    "mortality",
    "fertility rate", 
    "immunization",
    "GDP"
]

for q in queries:
    print(wb.series.info(q=q))

id                 value
-----------------  --------------------------------------------------------------------------------------------------------------------------
SH.DYN.MORT        Mortality rate, under-5 (per 1,000 live births)
SH.DYN.MORT.FE     Mortality rate, under-5, female (per 1,000 live births)
SH.DYN.MORT.MA     Mortality rate, under-5, male (per 1,000 live births)
SH.DYN.NCOM.FE.ZS  Mortality from CVD, cancer, diabetes or CRD between exact ages 30 and 70, female (%)
SH.DYN.NCOM.MA.ZS  Mortality from CVD, cancer, diabetes or CRD between exact ages 30 and 70, male (%)
SH.DYN.NCOM.ZS     Mortality from CVD, cancer, diabetes or CRD between exact ages 30 and 70 (%)
SH.DYN.NMRT        Mortality rate, neonatal (per 1,000 live births)
SH.STA.AIRP.FE.P5  Mortality rate attributed to household and ambient air pollution, age-standardized, female (per 100,000 female population)
SH.STA.AIRP.MA.P5  Mortality rate attributed to household and ambient air pollution, age-standardized, mal

In [76]:
indicators = {
    "SH.DYN.MORT": "Mortality rate, under-5 (per 1,000 live births)",
    "SP.DYN.TFRT.IN": "Fertility rate, total (births per woman)",
    "SH.IMM.IDPT": "Immunization, DPT (% of children ages 12-23 months)",
    "NY.GDP.PCAP.KD": "GDP per capita (constant 2015 US$)",
}

In [77]:
countries = ['ECU', 'BOL', 'BRA', 'CHL', 'COL', 'CRI', 'DOM', 'MEX', 'PRY', 'PER', 'URY']

df = wb.data.DataFrame(
    list(indicators.keys()),
    economy=countries,
    time=range(1975, 2023),
    labels=True
).reset_index()

print(df.head())
print(df.shape)

  economy       series             Country  \
0     URY  SH.DYN.MORT             Uruguay   
1     PER  SH.DYN.MORT                Peru   
2     PRY  SH.DYN.MORT            Paraguay   
3     MEX  SH.DYN.MORT              Mexico   
4     DOM  SH.DYN.MORT  Dominican Republic   

                                            Series  YR1975  YR1976  YR1977  \
0  Mortality rate, under-5 (per 1,000 live births)    55.0    54.1    51.9   
1  Mortality rate, under-5 (per 1,000 live births)   143.6   139.8   136.3   
2  Mortality rate, under-5 (per 1,000 live births)    73.1    71.8    70.3   
3  Mortality rate, under-5 (per 1,000 live births)    88.6    85.2    81.8   
4  Mortality rate, under-5 (per 1,000 live births)   104.8   100.9    97.1   

   YR1978  YR1979  YR1980  ...  YR2013  YR2014  YR2015  YR2016  YR2017  \
0    48.4    44.1    39.8  ...     9.7     9.3     8.9     8.4     8.0   
1   133.2   130.3   127.3  ...    17.8    17.2    16.6    16.1    15.7   
2    68.7    67.0    65.1  ...  

In [78]:
# Melt from wide to long
df_long = df.melt(
    id_vars=['economy', 'series', 'Country', 'Series'],
    var_name='Year',
    value_name='Value'
)

# Clean year column
df_long['Year'] = df_long['Year'].str.extract(r'(\d{4})').astype(int)

# Pivot to get indicators as columns
df_pivoted = df_long.pivot_table(
    index=['Country', 'economy', 'Year'],
    columns='Series',
    values='Value'
).reset_index()

print(df_pivoted.head())
print(df_pivoted.columns.tolist())
print(df_pivoted.shape)

Series  Country economy  Year  Fertility rate, total (births per woman)  \
0       Bolivia     BOL  1975                                     5.748   
1       Bolivia     BOL  1976                                     5.739   
2       Bolivia     BOL  1977                                     5.673   
3       Bolivia     BOL  1978                                     5.610   
4       Bolivia     BOL  1979                                     5.542   

Series  GDP per capita (constant 2015 US$)  \
0                              2081.069442   
1                              2127.832948   
2                              2182.675370   
3                              2176.909828   
4                              2130.754476   

Series  Immunization, DPT (% of children ages 12-23 months)  \
0                                                     NaN     
1                                                     NaN     
2                                                     NaN     
3                   

In [79]:
# - Brazil: removed due to scale incompatibility — 
#   GDP and health infrastructure diverge significantly from Ecuador
# - Chile: removed — regional outlier in child health outcomes 
#   by 1982, not a valid comparator
# - Costa Rica: removed — known regional leader in child 
#   mortality reduction, atypical trajectory
# - Uruguay: removed — higher income, lower mortality baseline, 
#   zero model weight confirmed exclusion

## Donor Pool Selection
Countries excluded: Brazil, Chile (scale incompatibility), Costa Rica, 
Uruguay (atypical regional trajectories). Final pool: Bolivia, Colombia, 
Dominican Republic, Mexico, Paraguay, Peru.

In [80]:
TREATED_UNIT = "Ecuador"
OUTCOME_VAR = "Mortality rate, under-5 (per 1,000 live births)"
PRE_TREATMENT_PERIOD = range(1982, 1992)
FULL_PERIOD = range(1982, 2023)
DONOR_POOL = ['Bolivia', 'Colombia', 'Dominican Republic', 
              'Mexico', 'Paraguay', 'Peru']
COVARIATES = [
    'Fertility rate, total (births per woman)',
    'GDP per capita (constant 2015 US$)'
]
SPECIAL_PREDICTORS_SETUP = [
    (OUTCOME_VAR, [1982, 1985, 1991], "mean"),
    ('Immunization, DPT (% of children ages 12-23 months)', [1982, 1991], "mean"),
    # ('School enrollment, primary (% gross)', [1982, 1991], "mean"),
]

## Model: Two-Step Synthetic Control
Step 1: Train on pre-treatment period (1982-1991) to learn V matrix.
Step 2: Generate full counterfactual path using learned V matrix.
Donor weights reported from Step 1 only.

In [81]:
# Step 1 — TRAIN on pre-treatment period only
dataprep_train = Dataprep(
    foo=df_pivoted,
    dependent=OUTCOME_VAR,
    unit_variable="Country",
    time_variable="Year",
    treatment_identifier=TREATED_UNIT,
    controls_identifier=DONOR_POOL,
    time_optimize_ssr=PRE_TREATMENT_PERIOD,
    predictors=COVARIATES,
    predictors_op="mean",
    time_predictors_prior=PRE_TREATMENT_PERIOD,
    special_predictors=SPECIAL_PREDICTORS_SETUP
)

synth_train = Synth()
synth_train.fit(dataprep=dataprep_train)
learned_v_matrix = synth_train.V
official_weights = synth_train.weights()
print("Donor Weights:\n", official_weights)

# Step 2 — GENERATE full counterfactual using learned V matrix
dataprep_full = Dataprep(
    foo=df_pivoted,
    dependent=OUTCOME_VAR,
    unit_variable="Country",
    time_variable="Year",
    treatment_identifier=TREATED_UNIT,
    controls_identifier=DONOR_POOL,
    time_optimize_ssr=FULL_PERIOD,
    predictors=COVARIATES,
    predictors_op="mean",
    time_predictors_prior=PRE_TREATMENT_PERIOD,
    special_predictors=SPECIAL_PREDICTORS_SETUP
)

synth_full = Synth()
synth_full.fit(dataprep=dataprep_full, custom_V=learned_v_matrix)

Donor Weights:
 Bolivia               0.083
Colombia              0.250
Dominican Republic    0.202
Mexico                0.144
Paraguay              0.204
Peru                  0.117
Name: weights, dtype: float64


In [82]:
synth_full = Synth()
synth_full.fit(dataprep=dataprep_full, custom_V=learned_v_matrix)

synth_full.path_plot(
    time_period=range(1982, 2023), 
    treatment_time=1992
)

synth_full.gaps_plot(treatment_time=1992)

C:\WBG\Python313\Lib\site-packages\pysyncon\base.py:101: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\WBG\Python313\Lib\site-packages\pysyncon\base.py:185: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Placebo Test
Running placebo tests on all donor countries to assess statistical 
significance of Ecuador's post-treatment gap.

In [83]:
placebo = PlaceboTest()
placebo.fit(
    dataprep=dataprep_train,
    scm=synth_train
)
placebo.gaps_plot(
    time_period=range(1982, 2023),
    treatment_time=1992
)

(1/6) Completed placebo test for Bolivia.
(2/6) Completed placebo test for Mexico.
(3/6) Completed placebo test for Peru.
(4/6) Completed placebo test for Colombia.
(5/6) Completed placebo test for Dominican Republic.
(6/6) Completed placebo test for Paraguay.
Calculating treated unit gaps.
Done.


C:\WBG\Python313\Lib\site-packages\pysyncon\utils.py:315: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [84]:
##visuals
matplotlib.use('Agg')

# After path plot
synth_full.path_plot(
    time_period=range(1982, 2023), 
    treatment_time=1992
)
plt.savefig('path_plot.png', dpi=150, bbox_inches='tight')
plt.show()

# After gaps plot
synth_full.gaps_plot(treatment_time=1992)
plt.savefig('gaps_plot.png', dpi=150, bbox_inches='tight')
plt.show()

# After placebo plot
placebo.gaps_plot(
    time_period=range(1982, 2023),
    treatment_time=1992
)
plt.savefig('placebo_plot.png', dpi=150, bbox_inches='tight')
plt.show()

C:\WBG\Python313\Lib\site-packages\pysyncon\base.py:101: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\wb642476\AppData\Local\Temp\ipykernel_25256\1359521448.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\WBG\Python313\Lib\site-packages\pysyncon\base.py:185: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\wb642476\AppData\Local\Temp\ipykernel_25256\1359521448.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\WBG\Python313\Lib\site-packages\pysyncon\utils.py:315: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\wb642476\AppData\Local\Temp\ipykernel_25256\1359521448.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Findings
The placebo test indicates Ecuador's post-treatment gap is not 
statistically distinguishable from donor placebos. The observed 
mortality decline reflects broader regional trends rather than 
intervention-specific effects. This null finding raises important 
questions about intervention attribution in contexts of widespread 
regional progress.